In [151]:
import pandas as pd
import re
import numpy as np
import torch
import torch.nn as nn
import sklearn
from transformers import AutoTokenizer, AutoModel, DistilBertForSequenceClassification
from tqdm import tqdm


import random
p = "/home/jupyter/datasphere/project/arXiv Dataset Metadata.json"




In [ ]:

data = []
chunks = pd.read_json(p,
                      lines=True, chunksize=50000)

for i, chunk in enumerate(chunks):
    # Clean & filter
    chunk = chunk[['id', 'title', 'abstract', 'update_date', 'categories']].dropna()
    chunk['title'] = chunk['title'].str.lower()
    chunk['abstract'] = chunk['abstract'].str.lower().apply(lambda x: re.sub(r"[^\w\d'\s]+", " ", x)) 
    chunk['created'] = pd.to_datetime(chunk['update_date'], errors='coerce')
    chunk['year'] = chunk['created'].dt.year
    recent = chunk[(chunk['year'] >= 2000) & (chunk['year'] <= 2025)]
    data.append(recent)

# Combine
data = pd.concat(data).reset_index(drop=True)
data = data[['id', 'abstract', 'title', 'categories']]

In [ ]:
      
vs = cats.explode().value_counts()
sklearn.utils.class_weight.compute_class_weight(vs.values)
weights = (max(vs) / vs).values
id_to_cat = dict(enumerate(vs.keys()))
cat_to_id = {cat: idx for idx, cat in id_to_cat.items()}
labels_ = cats.apply(lambda x: [cat_to_id[i] for i in x])

labels = np.zeros((len(labels_), len(cat_to_id)))
for i, label in enumerate(labels_):
    for j in label:
        labels[i, j] = 1

In [60]:
CATEGORIES = ['cs.LG', 'hep-ph', 'hep-th', 'quant-ph', 'cs.CV', 'cs.AI', 'gr-qc', 'astro-ph', 'cond-mat.mtrl-sci', 'cond-mat.mes-hall', 'math.MP', 'math-ph', 'cs.CL', 'cond-mat.str-el', 'cond-mat.stat-mech', 'astro-ph.CO', 'math.CO', 'stat.ML', 'astro-ph.GA', 'math.AP', 'astro-ph.SR', 'astro-ph.HE', 'math.PR', 'nucl-th', 'hep-ex', 'math.AG', 'math.OC', 'physics.optics', 'cs.IT', 'math.IT', 'cond-mat.supr-con', 'math.NT', 'math.DG', 'cond-mat.soft', 'math.NA', 'cs.RO', 'math.DS', 'cs.CR', 'cs.SY', 'math.FA', 'eess.SP', 'astro-ph.IM', 'astro-ph.EP', 'physics.flu-dyn', 'stat.ME', 'hep-lat', 'eess.SY', 'cs.NA', 'math.RT', 'eess.IV', 'nucl-ex', 'cs.DS', 'physics.comp-ph', 'stat.TH', 'math.ST', 'cond-mat.dis-nn', 'cs.NI', 'cs.DC', 'physics.chem-ph', 'math.GT', 'math.GR', 'math.CA', 'physics.soc-ph', 'cs.HC', 'cs.CY', 'physics.ins-det', 'cond-mat.quant-gas', 'physics.atom-ph', 'cs.SI', 'physics.app-ph', 'stat.AP', 'cs.IR', 'cs.SE', 'math.QA', 'math.RA', 'math.CV', 'physics.plasm-ph', 'eess.AS', 'cs.LO', 'cs.SD', 'math.AT', 'physics.bio-ph', 'nlin.CD', 'cond-mat.other', 'cs.NE', 'cs.DM', 'cond-mat', 'math.LO', 'math.AC', 'math.OA', 'cs.GT', 'nlin.SI', 'q-bio.PE', 'math.MG', 'cs.CC', 'q-bio.QM', 'physics.data-an', 'physics.gen-ph', 'q-bio.NC', 'math.SP', 'nlin.PS', 'cs.DB', 'math.SG', 'math.CT', 'cs.MA', 'stat.CO', 'physics.class-ph', 'cs.PL', 'cs.CE', 'physics.acc-ph', 'physics.med-ph', 'physics.geo-ph', 'cs.MM', 'nlin.AO', 'cs.GR', 'cs.CG', 'physics.ao-ph', 'physics.space-ph', 'math.KT', 'cs.AR', 'q-bio.BM', 'q-fin.EC', 'cs.ET', 'math.GN', 'cs.FL', 'cs.DL', 'physics.hist-ph', 'econ.GN', 'cs.PF', 'econ.EM', 'math.GM', 'physics.ed-ph', 'q-bio.MN', 'math.HO', 'q-fin.ST', 'q-bio.GN', 'econ.TH', 'physics.atm-clus', 'q-fin.MF', 'q-fin.GN', 'cs.SC', 'q-fin.CP', 'physics.pop-ph', 'q-fin.RM', 'q-bio.TO', 'chao-dyn', 'cs.MS', 'q-bio.CB', 'cs.OH', 'q-fin.PR', 'q-fin.PM', 'q-fin.TR', 'q-bio.SC', 'q-alg', 'nlin.CG', 'stat.OT', 'alg-geom', 'solv-int', 'q-bio.OT', 'q-bio', 'cs.OS', 'cmp-lg', 'dg-ga', 'patt-sol', 'adap-org', 'funct-an', 'mtrl-th', 'chem-ph', 'comp-gas', 'cs.GL', 'supr-con', 'atom-ph', 'acc-phys', 'plasm-ph', 'ao-sci', 'bayes-an']

In [119]:
# filter categories

num_examples = labels.sum(axis=0)
min_examples = 2000

cat_ids = np.arange(labels.shape[1])[num_examples > min_examples]

categories = [CATEGORIES[x] for x in cat_ids]
cat_to_id = {idx: item for idx, item in enumerate(categories)}

labels_filtered = labels[:, cat_ids]
passed_items = labels_filtered.sum(axis=-1) != 0
labels_filtered = labels_filtered[passed_items]
labels= labels_filtered
data = data[passed_items]

In [122]:
data.shape, labels.shape

((2703174, 5), (2703174, 151))

In [69]:
import json

with open("cat_to_id.json", "w") as f:
    f.write(json.dumps(cat_to_id))

np.save("arxiv_labels.npy", labels)
data.to_csv("/home/jupyter/datasphere/project/arxiv_items.csv")


In [117]:
import json
data = pd.read_csv("/home/jupyter/datasphere/project/arxiv_items.csv")
labels = np.load("arxiv_labels.npy")

/kernel/lib/python3.10/site-packages/ml_kernel/_vendor/IPython/core/interactiveshell.py:3553: DtypeWarning: Columns (2) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [152]:
from tqdm import tqdm

def prepare_index(labels, max_samples_per_class):
    samples_to_go = np.ones(labels.shape[1])  * max_samples_per_class
    index = np.arange(labels.shape[0])
    index_list = []
    
    for label in np.argsort(labels.sum(axis=0)):
        if samples_to_go[label] < 0:
            continue
        class_count = labels.sum(axis=0)[label]

        local_index = np.arange(labels.shape[0])
        candidates = local_index[labels[:, label] == 1]
        
        choosen_index = np.random.choice(candidates, replace=(class_count < samples_to_go[label]), size=int(samples_to_go[label]))    
        index_list.append(index[choosen_index])
        samples_to_go -= labels[choosen_index].sum(axis=0)
        
        ignore = samples_to_go <= 0
        
        to_continue = labels @ ignore == 0
        index = index[to_continue]
        labels = labels[to_continue]
    
    return np.concatenate(index_list)




(402633, 151)

In [164]:
from torch.utils.data import Dataset
import re


                                         

class TextDataset(Dataset):
    def __init__(self, data, labels, tokenizer, max_sequence=512, upsample=True, max_samples_per_class=8000, prob_to_add_abstract=0.2):
        if upsample:
            self.index = prepare_index(labels, max_samples_per_class)
        else:
            self.index = np.arange(labels.shape[0])
        np.random.shuffle(self.index)
        
        self.data = data.reset_index()
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_sequence = max_sequence
        self.prob_to_add_abstract = prob_to_add_abstract
                
    def __len__(self):
        return self.index.shape[0]
    
    def __getitem__(self, idx):
        idx = self.index[idx]
        
        target = self.labels[idx]
        title = self.data.iloc[idx]['title']
        abstract = self.data.iloc[idx]['abstract'] if random.random() < self.prob_to_add_abstract else ""
        
        item = tokenizer(title + "[SEP]" + abstract, padding="max_length", max_length=self.max_sequence, truncation=True)
        return torch.Tensor(target / target.sum(axis=-1)), item

In [142]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(data):
    labels, items = zip(*data)
    labels = torch.stack(labels)
    keys = ["input_ids", "attention_mask"]

    stacked = {
        tag: torch.stack([torch.tensor(obj[tag]) for obj in items], dim=0) for tag in keys
    }
    return labels, stacked

In [143]:
import wandb

wandb.login(key='9ccd1c7027333d858b318f3bcddf793897d3565c')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [144]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(data, labels, test_size=0.1)

In [145]:
def train(train_loader, valid_loader, model, criterion, optimizer, 
          device, num_epochs, model_path):
    """
    Function to train the model
    :param train_loader: Data loader for train dataset
    :param valid_loader: Data loader for validation dataset
    :param model: Model object
    :param criterion: Loss function
    :param optimizer: Optimizer
    :param device: CUDA or CPU
    :param num_epochs: Number of epochs
    :param model_path: Path to save the model
    """
    
    wandb.init(project="bert-paper-cls")
    best_loss = 1e8
    for i in range(num_epochs):
        print(f"Epoch {i+1} of {num_epochs}")
        valid_loss, train_loss = [], []
        model.train()
        # Train loop
        for batch_labels, batch_data in tqdm(train_loader):
            input_ids = batch_data["input_ids"]
            attention_mask = batch_data["attention_mask"]
            
            batch_labels = batch_labels.to(device)
            
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device, dtype=torch.bfloat16)
            input_ids = torch.squeeze(input_ids, 1)

            # Forward pass
            batch_output = model(input_ids, attention_mask).logits
            batch_output = torch.squeeze(batch_output)
            # Calculate loss
            ###batch_labels = batch_labels.type(torch.LongTensor)
            loss = criterion(batch_output, batch_labels)

            train_loss.append(loss.item())
            wandb.log({'train_loss': loss.item()})

            optimizer.zero_grad()
            # Backward pass
            loss.backward()
            # Gradient update step
            optimizer.step()
            torch.cuda.empty_cache()
        model.eval()
        # Validation loop
        for batch_labels, batch_data in tqdm(valid_loader):
            input_ids = batch_data["input_ids"]
            attention_mask = batch_data["attention_mask"]
            
            batch_labels = batch_labels.to(device)
            
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device, dtype=torch.bfloat16)
            input_ids = torch.squeeze(input_ids, 1)

            # Forward pass
            batch_output = model(input_ids, attention_mask).logits
            batch_output = torch.squeeze(batch_output)
            # Calculate loss
            ###batch_labels = batch_labels.type(torch.LongTensor)
            loss = criterion(batch_output, batch_labels)
            valid_loss.append(loss.item())

            wandb.log({'valid_loss': loss.item()})

            torch.cuda.empty_cache()
        t_loss = np.mean(train_loss)
        v_loss = np.mean(valid_loss)
        print(f"Train Loss: {t_loss}, Validation Loss: {v_loss}")
        if v_loss < best_loss:
            best_loss = v_loss
            # Save model if validation loss improves
            model.save_pretrained("model")
        print(f"Best Validation Loss: {best_loss}")

In [146]:
device = "cuda" # the device to load the model onto
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=len(cat_to_id)).to(device, dtype=torch.bfloat16)
model.requires_grad = False

for p in model.classifier.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5, eps=1e-8)

criterion = torch.nn.CrossEntropyLoss()

/home/jupyter/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [153]:
train_dataset = TextDataset(X_train, Y_train, tokenizer)
test_dataset = TextDataset(X_test, Y_test, tokenizer, upsample=False)


train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, collate_fn=collate_fn)

In [ ]:
train(train_loader, test_loader, model, criterion, optimizer, device, 1, "model.pt")